### MinMax or StandardScaler??

In [20]:
import copy
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
import statsmodels.api as sm
import itertools
from statsmodels.tsa.stattools import pacf, acf
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
from IPython.display import Image
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import classification_report, mean_squared_error,r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor

from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
import time as thetime
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import PassiveAggressiveRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import TheilSenRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
# import xgboost as xgb

from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
import multiprocessing
from multiprocessing import Pool # To run models in multiple threads simultaneously

from Functions import *

# To display tables in HTML output
from IPython.display import HTML, display

### Read in necessary data

In [53]:
import pandas as pd
sensors = pd.read_csv('../Cleaned_data/validsensors.csv')
weather = pd.read_csv('../Cleaned_data/weather_data_allyears.csv')
public_holidays = pd.read_csv('../Cleaned_data/publicholidays.csv')
features_near_sensors= pd.read_csv('../Cleaned_data/num_features_near_sensors_100.csv', index_col=0)
feature_subtypes_near_sensors = pd.read_csv('../Cleaned_data/feature_subtypes_near_sensors_100.csv', index_col=0).T

In [55]:
def label_hours (row):
    if row['time'] >6 and row['time'] <= 9:
        return 'morning rush hour'
    elif row['time'] >9 and row['time'] <= 12 :
        return 'morning'
    elif row['time'] >12 and row['time'] <= 15 :
        return 'afternoon'
    elif row['time'] >15 and row['time'] <= 18 :
        return 'afternoon rush hour'    
    elif row['time'] >18 and row['time'] <= 23 :
        return 'evening'    
    elif row['time'] == 23  or row['time'] <= 6 :
        return 'nighttime'        
sensors['time_of_day'] = sensors.apply (lambda row: label_hours(row), axis=1)

In [56]:
from Functions import *
sensors=convert_df_variables_to_dummy(sensors, ['time_of_day'])

### Remove outliers from sensors data
The model should predict normal footfall. Therefore any days that have extremely high or low footfall should be taken out of the training data. We don't actually want the model to try to predict footfall on unusual days, because the things that make the day unusual (like errors in the camera counters, or the presence of special events) are not captured in the input data.

Outliers are detected using the Median Absolute Deviation method.

In [57]:
sensors = remove_outliers(sensors)

I found 47746 outliers from 809202 days in total. Removing them leaves us with 761456 events


### Create formatted dataframes
3 versions: one with all basic features variables, one with feature variables subtypes and one with just the calendar variables and some weather variables

In [ ]:
df = create_formatted_df(sensors, features_near_sensors, feature_subtypes_near_sensors, public_holidays, 
                                  weather)
df = df.drop(['Pressure', 'bikes'],axis=1)
df_subtypes = create_formatted_df(sensors, features_near_sensors, feature_subtypes_near_sensors, public_holidays, weather,
                      use_subtypes = True)
df_basic = df[['sensor_id', 'hourly_counts', 'Temp', 'Rain', 'WindSpeed', 'public_holiday', 'Monday',  'Saturday',    
                    'Sunday', 'Thursday', 'Tuesday', 'Wednesday', 'August', 'December', 'February', 'January',
            'July', 'June','March', 'May',  'November', 'October', 'September',2012,2013,2014,2015,2016,2017,2018,2019,
              2020,2021,2022, 'Temp', 'Rain', 'morning rush hour', 'morning',  'afternoon rush hour', 'evening', 
               'nighttime' ,'WindSpeed', 'public_holiday']]

### Do some extra processing

In [17]:
### Filter to include just sensors which we know have quite complete data 
df = df[df['sensor_id'].isin([2,6,9,10,14,18])]
df.reset_index(inplace=True, drop = True)
df_subtypes = df_subtypes[df_subtypes['sensor_id'].isin([2,6,9,10,14,18])]
df_subtypes.reset_index(inplace=True, drop = True)
df_basic = df_basic[df_basic['sensor_id'].isin([2,6,9,10,14,18])]
df_basic.reset_index(inplace=True, drop = True)

## Modelling - Linear regression

#### Split into predictor and predictand variables

In [25]:
lr_df = perform_linear_regression(df)
lr_df_subtypes = perform_linear_regression(df_subtypes)
lr_df_basic = perform_linear_regression(df_basic)

Training score:  0.543964622464962
Test score:  0.5415566073808259
CV score:  0.5428561448037119
Training score:  0.5454180784631615
Test score:  0.543199177500779
CV score:  0.5472968486897556
Training score:  0.39246879938071444
Test score:  0.39064863056324106
CV score:  0.3920435650891776


In [28]:
lr_df_subtypes[0]['coefficients'].sort_values(ascending = False)[:30]

buildings_Community Use                   9.690818e+15
buildings_Retail - Showroom               8.478268e+15
buildings_Transport                       7.340566e+15
buildings_Parking - Private Covered       5.858391e+15
furniture_Barbeque                        4.267940e+15
buildings_Retail - Shop                   4.212140e+15
sensor_id                                 3.795369e+15
buildings_Manufacturing                   3.587163e+15
furniture_Information Pillar              3.492195e+15
furniture_Hoop                            3.155001e+15
furniture_Horse Trough                    2.826597e+15
landmarks_Leisure/Recreation              2.769425e+15
furniture_Drinking Fountain               1.732988e+15
landmarks_Health Services                 1.515826e+15
landmarks_Community Use                   1.135892e+15
landmarks_Place of Worship                1.133154e+15
buildings_Parking - Commercial Covered    1.092293e+15
furniture_Seat                            9.007813e+14
buildings_

In [8]:
# The predictor variables
Xfull = df.drop(['hourly_counts'], axis =1)

# The variable to be predicted
Yfull = df['hourly_counts'].values

# Store the predictors themselves in a list for future reference
predictors = list(df.drop(['hourly_counts'], axis=1).columns)

# Split data into training and test sets
X_train, X_test, Y_train, Y_test = \
    train_test_split(Xfull, Yfull, test_size=0.6666, random_state=123)
    
# Split off another training set now to train the hyperparameters using the two best models from above
X_validate, X_finaltest, Y_validate, Y_finaltest = \
    train_test_split(X_test, Y_test, test_size=0.5, random_state=123)

# Save copy of the X files with dates and then delete
X_finaltest_date = copy.deepcopy(X_finaltest)

In [16]:
def run_model(x):
    name, model_type = x # Unpack the tuple
    print(model_type)
    # See how long it takes to run this model
    start = thetime.time()
    # Use a pipeline to first scale the inputs (especially the weather)
    model = Pipeline (
        [ ('standardize', MinMaxScaler(feature_range = (0,1))), 
         (name, model_type())]
    )
    # Evaluate the pipeline (run the model)
    kfold = KFold(n_splits=10, random_state=7, shuffle = True)
    mse = cross_val_score(model, X_train, Y_train, cv=kfold, scoring = 'neg_mean_squared_error')
    r2 = cross_val_score(model, X_train, Y_train, cv=kfold, scoring = 'r2')
    
    # See how long it took (in seconds)
    runtime = int(thetime.time() - start)
    # Return the results, taking the median of the errors
    return (name, model, r2, np.median(r2), mse, np.median(mse), runtime)

### Find the best model  
Use k-fold cross validation to evaluate a range of regression algorithms on the training data. Use a pipeline for evaluation which first scales the (weather) data. Print the results and assess which models perform best.

The following models were trialled:

* Decision Tree
* Random Forest
* Extra Trees
* Dummy Regressor
* Elastic Net CV
* Passive Aggressive
* RANSAC
* SGD
* TheilSen (dropped in code below because it takes too long)
* K Neighbours
* LinearRegression
* XGBoost

In [ ]:
# Define a list of all the models to use
Models = {'LinearRegression': LinearRegression,'DecisionTree' : DecisionTreeRegressor,
          'RandomForest': RandomForestRegressor, 'ExtraTrees' : ExtraTreesRegressor,
          'DummyRegressor' :DummyRegressor, 'ElasticNetCV' : ElasticNetCV, 
          'PassiveAggressive' : PassiveAggressiveRegressor, #RANSAC': RANSACRegressor, # This one is terrible too
          'SGD': SGDRegressor, #'TheilSen': TheilSenRegressor, # Drop this - it isn't great and takes too long
          'KN': KNeighborsRegressor}#, 'XGBoost': xgb.XGBRegressor}
 
# Now just run each model, but do this in multiple processes simultaneously to save time    
# Now call that function simultaneously for each model
p = Pool(processes=None) # A pool of processes (one for each core)
results = p.map(run_model, [(name, model_type) for name, model_type in Models.items()])

# Sort the results by median mse (that's item 5 in the tuple)
results.sort(key=lambda x: x[5], reverse=True)

# Put the results in a nice dictionary and print them
results_dict = {}
txt = "<table><thead><td>Name</td><td>Median R2</td><td>Median MSE</td><td>runtime (sec)</td></thead>"
for name, model, all_r2, r2, all_mse, mse, runtime in results:
    txt += "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(name, r2, mse, runtime)
    results_dict[name] = (model, all_r2, r2, all_mse, mse, runtime)
txt += "</table>"
display(HTML(txt)) # print as html

<class 'sklearn.linear_model._base.LinearRegression'><class 'sklearn.tree._classes.DecisionTreeRegressor'><class 'sklearn.ensemble._forest.RandomForestRegressor'><class 'sklearn.ensemble._forest.ExtraTreesRegressor'><class 'sklearn.linear_model._coordinate_descent.ElasticNetCV'><class 'sklearn.dummy.DummyRegressor'><class 'sklearn.linear_model._passive_aggressive.PassiveAggressiveRegressor'><class 'sklearn.neighbors._regression.KNeighborsRegressor'><class 'sklearn.linear_model._stochastic_gradient.SGDRegressor'>










In [ ]:
min_mse = min([mse for (name, model, all_r2, r2, all_mse, mse, runtime) in results])
               
x =  [ name for (name, model, all_r2, r2, all_mse, mse, runtime) in results]
y1 = [ mse-min_mse   for (name, model, all_r2, r2, all_mse, mse, runtime) in results]
y2 = [ r2 if r2 > 0 else 0 for (name, model, all_r2, r2, all_mse, mse, runtime) in results]

fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(15, 7))

ax1.set_title("MSE")
#ax1.invert_yaxis()
ax1.bar(range(len(x)), y1)
ax1.set_xticks(range(len(x)))
ax1.set_xticklabels(x, rotation=90)
ax1.set_ylim([27000000000, 29000000000])

ax2.set_title("R^2")
ax2.bar(range(len(x)), y2)
ax2.set_xticks(range(len(x)))
ax2.set_xticklabels(x, rotation=90)

plt.show()

#del x,y1, y2

In [ ]:
## Set up a dictionary containing the hyperparameters we want to tune
hyperparameters_rf = { 'randomforestregressor__max_features' : ['auto', 'sqrt', 'log2'],
                  'randomforestregressor__max_depth': [None, 5, 3, 1]}
# hyperparameters_xgb = {'xgbregressor__max_depth': range(1, 11, 2),
#                    'xgbregressor__n_estimators' : range(50, 400, 50),
#                    'xgbregressor__learning_rate' : [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3]}
hyperparameters_lr = {}

# Set up the pipeline containing the scalers
pipeline_rf = make_pipeline(MinMaxScaler(feature_range = (0,1)), 
                         RandomForestRegressor(n_estimators=100))
# pipeline_xgb = make_pipeline(MinMaxScaler(feature_range = (0,1)),
#                          xgb.XGBRegressor(n_estimators=100))
pipeline_lr = make_pipeline(MinMaxScaler(feature_range = (0,1)),
                         LinearRegression())

# Store the scores in a results dictionary (and print them)
final_results = {}
for model_values in [(pipeline_rf,  hyperparameters_rf,  'RandomForest'),
#                      (pipeline_xgb, hyperparameters_xgb, 'XGBoost'),
                     (pipeline_lr,  hyperparameters_lr,  'LinearRegression')]:
    
    clf = GridSearchCV(model_values[0], model_values[1], 
                       #cv = None, # Cross-validation method. None means default (3-fold)
                       cv = 10, # positive intiger means k-fold (e.g. 10-fold)
                       #scoring  = 'neg_mean_squared_error', # MSE to calculate score
                       scoring  = 'r2', # MSE to calculate score
                       n_jobs=multiprocessing.cpu_count()) # Run on multiple cores
    
    #clf = GridSearchCV(model_values[0], model_values[1], cv = 10, scoring  = 'r2')
    clf.fit(X_validate, Y_validate)
    name = model_values[2]
    final_results[name] = clf
    print ("Hyperparameter results for {}".format(name))
    print ("\tBest Score: {}".format(clf.best_score_))
    print ("\tBest params: {}".format(clf.best_params_))

#### Ridge with GridSearchCV

In [ ]:
parameters = {'alpha': list(range(10)), 'fit_intercept': [True, False], 
              'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']}

# define the model/ estimator
model = Ridge(max_iter = 100000)

# define the grid search
ridge= GridSearchCV(model, parameters,cv=5)

#fit the grid search
ridge.fit(X_train,Y_train)

# best estimator
print(ridge.best_estimator_)

# best model
best_model = ridge.best_estimator_
best_model.fit(X_train,y_train)

In [ ]:
print('Training score: ', best_model.score(X_train, y_train))
print('Test score: ', best_model.score(X_test, y_test))
print('CV score: ', (cross_val_score(best_model, X_train, y_train)).mean())

#### Lasso with GridSearchCV

In [ ]:
parameters = {'alpha': np.logspace(-4, 4, 10), 'fit_intercept': [True, False]}

# define the model/ estimator
model = Lasso(max_iter = 100000)

# define the grid search
lasso= GridSearchCV(model, parameters,cv=5)

#fit the grid search
lasso.fit(X_train,y_train)

# best estimator
print(lasso.best_estimator_)

# best model
best_model = lasso.best_estimator_
best_model.fit(X_train,y_train)

In [ ]:
print('Training score: ', best_model.score(X_train, y_train))
print('Test score: ', best_model.score(X_test, y_test))
print('CV score: ', (cross_val_score(best_model, X_train, y_train)).mean())

#### Decision Tree Regressor with GridSearchCV

In [ ]:
dtr_params = {
    'max_depth': list(range(1, 11))+[None],
    'max_features': [None, 1, 2, 3],
    'min_samples_split': [2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50],
    'ccp_alpha': [0, 0.001, 0.005, 0.009, 0.01, 0.05]
}


# set the gridsearch
model = DecisionTreeRegressor()
dtr_gs = GridSearchCV(model, dtr_params, cv=5, verbose=1, n_jobs=2)
dtr_gs.fit(X_train, y_train)
print(dtr_gs.best_params_)
best_model = dtr_gs.best_estimator_

In [ ]:
print('Training score: ', best_model.score(X_train, y_train))
print('Test score: ', best_model.score(X_test, y_test))
print('CV score: ', (cross_val_score(best_model, X_train, y_train)).mean())

In [ ]:
train = pd.concat([all_sen_2012, all_sen_2013, all_sen_2014, all_sen_2015, all_sen_2016], axis = 0, sort = True)
test = pd.concat([all_sen_2018, all_sen_2017], axis = 0, sort = True)

X_train = train.copy()
y_train = X_train.pop('daily_avg_counts')

X_test = test.copy()
y_test = X_test.pop('daily_avg_counts')

In [ ]:
#standardize
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

### Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
print('Training score: ', model.score(X_train, y_train))
print('Test score: ', model.score(X_test, y_test))
print('CV score: ', (cross_val_score(model, X_train, y_train)).mean())

In [ ]:
predictions = model.predict(X_test)
residuals = pd.DataFrame(predictions, y_test)
residuals.reset_index(inplace = True)
residuals.rename({'daily_avg_counts': 'y_test', 0: 'predictions'}, axis = 1, inplace = True)
residuals['residuals'] = residuals.y_test - residuals.predictions
residuals

In [ ]:
(mean_squared_error(y_test, predictions))**0.5

### Ridge with GridSearchCV

In [ ]:
parameters = {'alpha': np.logspace(-20, 20, 50), 'fit_intercept': [True, False], 
              'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']}

# define the model/ estimator
model = Ridge(max_iter = 100000)

# define the grid search
ridge= GridSearchCV(model, parameters,cv=5)

#fit the grid search
ridge.fit(X_train,y_train)

# best estimator
print(ridge.best_estimator_)

# best model
best_model = ridge.best_estimator_
best_model.fit(X_train,y_train)

In [ ]:
# Ridge(alpha=4714.8663634573895, copy_X=True, fit_intercept=True,
#       max_iter=100000, normalize=False, random_state=None, solver='lsqr',
#       tol=0.001)

print('Training score: ', best_model.score(X_train, y_train))
print('Test score: ', best_model.score(X_test, y_test))
print('CV score: ', (cross_val_score(best_model, X_train, y_train)).mean())

### Lasso with GridSearchCV

In [ ]:
parameters = {'alpha': np.logspace(-4, 4, 10), 'fit_intercept': [True, False]}

# define the model/ estimator
model = Lasso(max_iter = 100000)

# define the grid search
lasso= GridSearchCV(model, parameters,cv=5)

#fit the grid search
lasso.fit(X_train,Y_train)

# best estimator
print(lasso.best_estimator_)

# best model
best_model = lasso.best_estimator_
best_model.fit(X_train,y_train)

Lasso(alpha=21.54434690031882, copy_X=True, fit_intercept=True, max_iter=100000,
      normalize=False, positive=False, precompute=False, random_state=None,
      selection='cyclic', tol=0.0001, warm_start=False)

In [ ]:
#sns.set_style('ticks')
sns.set(font_scale = 2)
fig, ax = plt.subplots()
# the size of A4 paper
fig.set_size_inches(16, 12)
sns.barplot(x='importance', y='feature', data=fi[-15:], orient='h', palette = 'rocket', saturation=0.7)  
ax.set_title("Feature Importance", fontsize=40, y=1.01)
ax.set_xlabel('Importance', fontsize = 30)
ax.set_ylabel('Feature', fontsize = 30)

In [ ]:
#predicted y values
predictions = best_model.predict(X_test)

#residuals (or error between predictions and actual)
residuals = y_test - predictions

sns.axes_style(style='white')
sns.set(font_scale = 2)
fig, ax = plt.subplots()
fig.set_size_inches(16, 12)
ax = sns.regplot(x="predictions", y="y_test", data= residuals_df,  scatter_kws = {'color': 'lightsalmon'}, 
                 line_kws = {'color': 'darksalmon'})
ax.set_xlabel('Predicted', fontsize = 30)
ax.set_ylabel('Actual', fontsize = 30)


### Decision Tree Regressor

In [ ]:
dtr_params = {
    'max_depth': [5, 10, 15, 20,50],
    'max_features': [10, 50, None],
    'min_samples_split': [10, 20, 40, 50, 70],
    'ccp_alpha': [0.001, 0.005, 0.01, 0.1]}

# set the gridsearch
model = DecisionTreeRegressor()

dtr_gs = GridSearchCV(model, dtr_params, cv=5, verbose=1, n_jobs=2)
dtr_gs.fit(X_train, y_train)
print(dtr_gs.best_params_)
best_model = dtr_gs.best_estimator_

In [ ]:
print('Training score: ', best_model.score(X_train, y_train))
print('Test score: ', best_model.score(X_test, y_test))
print('CV score: ', (cross_val_score(best_model, X_train, y_train)).mean())